In [ ]:
######################### 
# 나비야넷 크롤링
#########################

In [1]:
import re
import time
import os
from datetime import datetime
from urllib.parse import urljoin, urlparse, parse_qs, urlencode, urlunparse

import requests
import pandas as pd
from bs4 import BeautifulSoup as bs
from tqdm.notebook import tqdm

In [10]:
BASE = "https://www.naviya.net"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.8",
})

def get_soup(url, sleep_sec=0.1, max_retries=2):
    last_exc = None
    for attempt in range(max_retries + 1):
        try:
            r = session.get(url, timeout=25)
            # 404 등은 여기서 raise
            r.raise_for_status()
            if sleep_sec:
                time.sleep(sleep_sec)
            return bs(r.text, "html.parser"), r.text
        except requests.HTTPError as e:
            last_exc = e
            # 404는 재시도해도 소용없으니 즉시 throw
            if e.response is not None and e.response.status_code == 404:
                raise
            # 그 외(500/502/429 등)는 재시도
            time.sleep(0.5 * (attempt + 1))
        except Exception as e:
            last_exc = e
            time.sleep(0.5 * (attempt + 1))
    raise last_exc

In [11]:
RAW_COLS = [
    "source_site", "crawled_at", "listing_url", "detail_url", "external_id",
    "name_raw", "category_raw", "address_raw", "map_url_raw", "status_raw"
]

def now_kst_str():
    return time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

In [12]:
TARGET_BANNERTAB2 = {"99@116@": "강남/잠실", "200@210@": "비강남(동/서/북서울)"}

def normalize_url(u: str) -> str:
    u = (u or "").strip()
    full = urljoin(BASE + "/", u)

    # ✅ /board.php 로 들어오면 /bbs/board.php로 교정
    p = urlparse(full)
    if p.path.endswith("/board.php") and not p.path.startswith("/bbs/"):
        full = full.replace("/board.php", "/bbs/board.php")

    return full

def collect_listing_urls():
    seeds = [
        BASE + "/",
        urljoin(BASE, "/bbs/board.php?bo_table=b49&bannertab2=99@116@"),
        urljoin(BASE, "/bbs/board.php?bo_table=b49&bannertab2=200@210@"),
    ]

    found = set()

    for s in seeds:
        soup, _ = get_soup(s)
        for a in soup.select('a[href*="board.php"]'):
            href = a.get("href", "")
            if not href:
                continue

            full = normalize_url(href)
            if "bo_table=b49" not in full:
                continue

            qs = parse_qs(urlparse(full).query)
            bt = qs.get("bannertab2", [None])[0]
            if bt not in TARGET_BANNERTAB2:
                continue

            found.add(full)

    # 탭 기본 URL 강제 포함(정규화된 /bbs/board.php)
    found.add(urljoin(BASE, "/bbs/board.php?bo_table=b49&bannertab2=99@116@"))
    found.add(urljoin(BASE, "/bbs/board.php?bo_table=b49&bannertab2=200@210@"))

    return sorted(found)

listing_urls = collect_listing_urls()
print("listing_urls:", len(listing_urls))
listing_urls[:10]

listing_urls: 73


['https://www.naviya.net/bbs/board.php?bo_table=b49&bannertab2=200%40210%40&page=2',
 'https://www.naviya.net/bbs/board.php?bo_table=b49&bannertab2=200%40210%40&page=3',
 'https://www.naviya.net/bbs/board.php?bo_table=b49&bannertab2=200%40210%40&page=4',
 'https://www.naviya.net/bbs/board.php?bo_table=b49&bannertab2=200%40210%40&page=5',
 'https://www.naviya.net/bbs/board.php?bo_table=b49&bannertab2=200%40210%40&page=6',
 'https://www.naviya.net/bbs/board.php?bo_table=b49&bannertab2=200%40210%40&page=7',
 'https://www.naviya.net/bbs/board.php?bo_table=b49&bannertab2=200@210@',
 'https://www.naviya.net/bbs/board.php?bo_table=b49&bannertab2=99%40116%40&page=2',
 'https://www.naviya.net/bbs/board.php?bo_table=b49&bannertab2=99%40116%40&page=3',
 'https://www.naviya.net/bbs/board.php?bo_table=b49&bannertab2=99%40116%40&page=4']

In [13]:
def with_page(url: str, page: int) -> str:
    """listing_url에 page 파라미터를 세팅한 URL을 만든다."""
    p = urlparse(url)
    qs = parse_qs(p.query)
    qs["page"] = [str(page)]
    new_query = urlencode(qs, doseq=True)

    return urlunparse((p.scheme, p.netloc, p.path, p.params, new_query, p.fragment))

In [14]:
WR_ID_RE = re.compile(r"[?&]wr_id=(\d+)")

def parse_listing_page(listing_url: str):
    soup, _ = get_soup(listing_url)

    rows = []
    for a in soup.select('a[href*="wr_id="]'):
        href = a.get("href", "")
        m = WR_ID_RE.search(href)
        if not m:
            continue

        external_id = m.group(1)
        detail_url = urljoin(listing_url, href)
        title = a.get_text(" ", strip=True) or None

        rows.append({
            "listing_url": listing_url,
            "detail_url": detail_url,
            "external_id": external_id,
            "name_raw": title,
        })

    # 페이지 내 중복 제거(merge 방식)
    by_id = {}
    for r in rows:
        k = r["external_id"]
        if k not in by_id:
            by_id[k] = r
        else:
            if (not by_id[k].get("name_raw")) and r.get("name_raw"):
                by_id[k]["name_raw"] = r["name_raw"]
            # detail_url이 다를 수도 있으니 최신으로 갱신
            if r.get("detail_url"):
                by_id[k]["detail_url"] = r["detail_url"]

    return list(by_id.values())


In [15]:
from urllib.parse import urlparse, parse_qs

def parse_detail_page(detail_url: str):
    soup, _ = get_soup(detail_url)

    # ✅ address_raw: hidden input wr_59에서 확정 추출
    address_raw = None
    inp = soup.select_one('input[name="wr_59"]')
    if inp and inp.get("value"):
        address_raw = inp.get("value").strip()

    # ✅ map_url_raw: 좌표 미확보 → 상세 URL을 지도 근거로 사용
    map_url_raw = detail_url

    return {
        "address_raw": address_raw,
        "map_url_raw": map_url_raw,
    }

def build_raw_row(listing_url, detail_url, external_id, name_raw_from_list=None):
    d = parse_detail_page(detail_url)

    return {
        "source_site": "naviya.net",
        "crawled_at": now_kst_str(),
        "listing_url": listing_url,
        "detail_url": detail_url,
        "external_id": external_id,
        "name_raw": name_raw_from_list,
        "category_raw": None,          # ✅ 의도적으로 채우지 않음
        "address_raw": d.get("address_raw"),
        "map_url_raw": d.get("map_url_raw"),
        "status_raw": None,
    }

In [16]:
def crawl_listings(listing_urls, max_pages=30):
    collected = {}

    for base_url in listing_urls:
        prev_page_ids = None

        for page in range(1, max_pages + 1):
            page_url = with_page(base_url, page)

            try:
                items = parse_listing_page(page_url)
            except requests.HTTPError as e:
                # ✅ 404 등은 스킵하고 다음으로
                code = e.response.status_code if e.response is not None else None
                if code == 404:
                    break  # 이 base_url에선 더 이상 페이지가 없다고 보고 중단
                continue

            if not items:
                break

            page_ids = tuple(sorted({it["external_id"] for it in items}))
            if prev_page_ids is not None and page_ids == prev_page_ids:
                break
            prev_page_ids = page_ids

            for it in items:
                uid = it["external_id"]
                if uid not in collected:
                    collected[uid] = it
                else:
                    if (not collected[uid].get("name_raw")) and it.get("name_raw"):
                        collected[uid]["name_raw"] = it["name_raw"]
                    if it.get("detail_url"):
                        collected[uid]["detail_url"] = it["detail_url"]

    return list(collected.values())

all_items = crawl_listings(listing_urls, max_pages=30)
print("unique items:", len(all_items))
pd.DataFrame(all_items).head(5)


unique items: 352


,listing_url,detail_url,external_id,name_raw
0,https://www.naviya.net/bbs/board.php?bo_table=...,https://www.naviya.net/bbs/board.php?bo_table=...,45961,강서 정스웨디시
1,https://www.naviya.net/bbs/board.php?bo_table=...,https://www.naviya.net/bbs/board.php?bo_table=...,49912,중랑1인샵 에르메스
2,https://www.naviya.net/bbs/board.php?bo_table=...,https://www.naviya.net/bbs/board.php?bo_table=...,47942,남쌤1인샵 온결
3,https://www.naviya.net/bbs/board.php?bo_table=...,https://www.naviya.net/bbs/board.php?bo_table=...,42283,양천 노블테라피
4,https://www.naviya.net/bbs/board.php?bo_table=...,https://www.naviya.net/bbs/board.php?bo_table=...,50705,사당[청담스웨디시] 신규오픈 ⭐ 사당역 4번출구 스웨디시 전문샵 ⭐✨ 최고의 시설과...


In [18]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm 

def build_row_safe(it):
    try:
        return build_raw_row(
            listing_url=it["listing_url"],
            detail_url=it["detail_url"],
            external_id=it["external_id"],
            name_raw_from_list=it.get("name_raw"),
        )
    except Exception as e:
        # 실패해도 행은 남겨서 나중에 재시도/진단 가능
        return {
            "source_site": "naviya.net",
            "crawled_at": now_kst_str(),
            "listing_url": it.get("listing_url"),
            "detail_url": it.get("detail_url"),
            "external_id": it.get("external_id"),
            "name_raw": it.get("name_raw"),
            "category_raw": None,
            "address_raw": None,
            "map_url_raw": it.get("detail_url"),
            "status_raw": f"ERROR: {type(e).__name__}",
        }

max_workers = 8  # 6~10 권장
rows = []

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = [ex.submit(build_row_safe, it) for it in all_items]
    for f in tqdm(as_completed(futures), total=len(futures)):
        rows.append(f.result())

df_raw = pd.DataFrame(rows, columns=RAW_COLS)

display(df_raw.head(5))
print("address_raw missing ratio:", df_raw["address_raw"].isna().mean())


100%|██████████| 352/352 [00:37<00:00,  9.37it/s]


,source_site,crawled_at,listing_url,detail_url,external_id,name_raw,category_raw,address_raw,map_url_raw,status_raw
0,naviya.net,2026-01-26 12:56:13,https://www.naviya.net/bbs/board.php?bo_table=...,https://www.naviya.net/bbs/board.php?bo_table=...,42283,양천 노블테라피,None,서울특별시 양천구 신정동 1031-9 (중앙로 262) / 신정네거리역 4번출구 도보1분,https://www.naviya.net/bbs/board.php?bo_table=...,None
1,naviya.net,2026-01-26 12:56:13,https://www.naviya.net/bbs/board.php?bo_table=...,https://www.naviya.net/bbs/board.php?bo_table=...,50705,사당[청담스웨디시] 신규오픈 ⭐ 사당역 4번출구 스웨디시 전문샵 ⭐✨ 최고의 시설과...,None,서울특별시 관악구 남현동 1061-24/서울특별시 관악구 과천대로 947-2/사당역...,https://www.naviya.net/bbs/board.php?bo_table=...,None
2,naviya.net,2026-01-26 12:56:13,https://www.naviya.net/bbs/board.php?bo_table=...,https://www.naviya.net/bbs/board.php?bo_table=...,48635,마포.공덕[NOW] ⭐나우⭐ ღ 재방 많은 샵 ღ 힐링케어 해드립니다❣️,None,서울특별시 마포구 도화동 1-12 (새창로 40) / 공덕역 10번출구 도보2분,https://www.naviya.net/bbs/board.php?bo_table=...,None
3,naviya.net,2026-01-26 12:56:13,https://www.naviya.net/bbs/board.php?bo_table=...,https://www.naviya.net/bbs/board.php?bo_table=...,47942,남쌤1인샵 온결,None,서울특별시 성동구 도선동 58-1 (왕십리로 326)/왕십리역 1번출구 도보5분 (...,https://www.naviya.net/bbs/board.php?bo_table=...,None
4,naviya.net,2026-01-26 12:56:13,https://www.naviya.net/bbs/board.php?bo_table=...,https://www.naviya.net/bbs/board.php?bo_table=...,45961,강서 정스웨디시,None,서울특별시 강서구 마곡동 797-10 (마곡동로4길 15) / 발산역 9번출구 도보...,https://www.naviya.net/bbs/board.php?bo_table=...,None


address_raw missing ratio: 0.0


In [19]:
out_dir = "../data"
os.makedirs(out_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = os.path.join(out_dir, f"naviya_raw_{timestamp}.csv")

df_raw.to_csv(out_path, index=False, encoding="utf-8-sig")
print("saved:", os.path.abspath(out_path))

saved: c:\workspace\2025-sbsDSJA\TeamProject\data\naviya_raw_20260126_130108.csv


In [20]:
##############
# 테스트
##############
print("rows:", len(df_raw), "unique external_id:", df_raw["external_id"].nunique())
print(df_raw["category_raw"].value_counts(dropna=False))
print("name_raw missing:", df_raw["name_raw"].isna().mean())
print("address_raw missing:", df_raw["address_raw"].isna().mean())
print("map_url_raw unique:", df_raw["map_url_raw"].nunique())

rows: 352 unique external_id: 352
category_raw
None    352
Name: count, dtype: int64
name_raw missing: 0.0
address_raw missing: 0.0
map_url_raw unique: 352
